<a href="https://colab.research.google.com/github/trainocate-japan/openai_api_app/blob/main/chapter3/exercise/%E7%AC%AC3%E7%AB%A0%20(%E6%BC%94%E7%BF%92)%20%E7%94%BB%E5%83%8F%E7%94%9F%E6%88%90%E3%83%81%E3%83%A3%E3%83%83%E3%83%88%E3%83%9C%E3%83%83%E3%83%88_GPTImage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 演習 3-1-7

In [ ]:
# パッケージインストール
# !pip install openai
# !pip install tiktoken
# !pip install openai
# streamlit関連パッケージのインストール
!pip install streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!which cloudflared && cloudflared --version

In [ ]:
# app.pyの中身を実装
%%writefile app.py
from openai import OpenAI
import streamlit as st
from PIL import Image
import base64
import requests
import io

## APIキーを設定したclientオブジェクト生成
client =

st.title("画像を生成するボット")
user_input = st.text_input("生成したい画像のキーワードを入力してください:")

if st.button("送信"):
    ## completionsを用いて、user_inputを英語に翻訳する
    english_keywords = client.chat.completions.create(
        model= ,
        messages=
    )

    ## client.images.generateメソッドを用いて画像を生成する
    response = client.images.generate(
        model= ,
        prompt= ,
        size="1024x1024",
        n=1
    )

    st.text_area("ボットの応答", "画像を生成しました")
    image_base64 = response.data[0].b64_json
    image_bytes = base64.b64decode(image_base64)
    image = Image.open(io.BytesIO(image_bytes))
    st.image(image, caption='Created by AI',use_column_width=True)

In [ ]:
# streamlitのrunコマンドでapp.pyを立ち上げ、localtunnelを用いてアプリ公開
# !streamlit run app.py & sleep 3 && npx localtunnel --port 8501
!streamlit run app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.headless true \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --server.fileWatcherType none \
  > /tmp/st.log 2>&1 &
!for i in {1..60}; do curl -fsS http://localhost:8501/healthz && echo "Streamlit is up" && break || sleep 1; done

# トンネル起動（ログにURLが出る）
!cloudflared tunnel --url http://localhost:8501 --no-autoupdate > /tmp/cf.log 2>&1 &

# URLがログに出るまで最大60秒待って抽出
!for i in {1..60}; do \
  URL=$(grep -o "https://[0-9a-z.-]*trycloudflare.com" -m 1 /tmp/cf.log); \
  if [ -n "$URL" ]; then echo "PUBLIC URL: $URL"; break; fi; \
  sleep 1; \
done

In [ ]:
# app.pyの中身を実装
%%writefile app.py
from openai import OpenAI
import streamlit as st
from PIL import Image
import base64
import requests
import io

client = OpenAI(api_key="your api key")

st.title("画像を生成するボット")
user_input = st.text_input("生成したい画像のキーワードを入力してください:")

if st.button("送信"):
    english_keywords = client.chat.completions.create(
        model= "gpt-4o-mini",
        messages=[{"role": "system", "content": "あなたは翻訳家です。日本語を英語に翻訳してください。"},
                  {"role": "user", "content": user_input}
                  ]
    )

    response = client.images.generate(
        model="gpt-image-1.5",
        prompt=english_keywords.choices[0].message.content,
        size="1024x1024",
        n=1
    )

    st.text_area("ボットの応答", "画像を生成しました")
    image_base64 = response.data[0].b64_json
    image_bytes = base64.b64decode(image_base64)
    image = Image.open(io.BytesIO(image_bytes))
    st.image(image, caption='Created by AI',use_column_width=True)

    ## ファイルパスを指定して画像を保存する
    image.save(" ")
    st.text_area("ボットの応答", "画像を保存しました")

In [ ]:
# streamlitのrunコマンドでapp.pyを立ち上げ、localtunnelを用いてアプリ公開
# !streamlit run app.py & sleep 3 && npx localtunnel --port 8501
!streamlit run app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.headless true \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --server.fileWatcherType none \
  > /tmp/st.log 2>&1 &
!for i in {1..60}; do curl -fsS http://localhost:8501/healthz && echo "Streamlit is up" && break || sleep 1; done

# トンネル起動（ログにURLが出る）
!cloudflared tunnel --url http://localhost:8501 --no-autoupdate > /tmp/cf.log 2>&1 &

# URLがログに出るまで最大60秒待って抽出
!for i in {1..60}; do \
  URL=$(grep -o "https://[0-9a-z.-]*trycloudflare.com" -m 1 /tmp/cf.log); \
  if [ -n "$URL" ]; then echo "PUBLIC URL: $URL"; break; fi; \
  sleep 1; \
done